# Data Download

Downloads all structural data needed for the connectome network analysis.
Restricted to axon-proofread neurons and synapses between them.

Run this notebook once before any analysis notebooks.

In [9]:
import os
import microns_datacleaner as mic
import microns_datacleaner.filters as fl

VERSION  = 1718
DATADIR  = "data"
SYNAPSE_FILE = "proofread_synapses"

## 1. Nucleus tables

In [10]:
cleaner = mic.MicronsDataCleaner(datadir=DATADIR, version=VERSION, download_policy='minimum')

raw_dir = os.path.join(os.getcwd(), DATADIR, str(VERSION), "raw")
existing = {os.path.splitext(f)[0] for f in os.listdir(raw_dir) if f.endswith(".csv")} if os.path.isdir(raw_dir) else set()
missing  = set(cleaner.tables_2_download) - existing

if missing:
    print(f"Downloading {len(missing)} missing nucleus table(s): {missing}")
    cleaner.download_nucleus_data()
else:
    print("Nucleus tables already present — skipping.")

## 2. Proofread neurons

In [11]:
units, segments = cleaner.process_nucleus_data()
proofread = fl.filter_neurons(units, proofread='ax_clean')
print(f"Total units:       {len(units)}")
print(f"Proofread neurons: {len(proofread)}")

Transform positions: 100%|██████████| 347/347 [00:00<00:00, 69193.85it/s]

Total units:       342
Proofread neurons: 6


## 3. Synapses between proofread neurons

In [12]:
synapse_path = os.path.join(os.getcwd(), DATADIR, str(VERSION), "raw", f"{SYNAPSE_FILE}.csv")

if os.path.exists(synapse_path):
    print("Synapse file already exists — skipping download.")
else:
    ids = proofread['pt_root_id']
    print(f"Downloading synapses for {len(ids)} proofread neurons...")
    cleaner.download_synapse_data(ids, ids)
    cleaner.merge_synapses(syn_table_name=SYNAPSE_FILE)
    print("Done.")

print(f"Synapse file: {synapse_path}")

100%|██████████| 1/1 [00:00<00:00,  1.36it/s]

Done.
Synapse file: c:\Users\leona\AppData\Local\Programs\Python\Neuro-1\data\1718\raw\proofread_synapses.csv
